# Analyze

In [1]:
import pandas as pd
import os
import math

In [2]:
df = pd.read_csv("room_amenities.csv")
df

,room_type_id,room_amenities
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':..."
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':..."
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':..."
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':..."
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':..."
...,...,...
10061,10062,"[{'amenity': 'Diện tích phòng: 16 m²', 'type':..."
10062,10063,"[{'amenity': 'Diện tích phòng: 16 m²', 'type':..."
10063,10064,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':..."
10064,10065,"[{'amenity': 'Diện tích phòng: 15 m²', 'type':..."


In [3]:
import ast

# 1. Đọc lại file gốc, không qua xử lý trung gian nào
df = pd.read_csv("room_amenities.csv")  # nhớ đúng đường dẫn

def classify(value):
    # --- NULL / rỗng ---
    if pd.isna(value):
        return "null"
    s = str(value).strip()
    if s == "" or s.lower() == "null" or s == "[]":
        return "null"
    
    # --- Try to parse string to object Python ---
    try:
        obj = ast.literal_eval(s)
    except Exception:
        # Have data but cannot parse  -> unstructured
        return "unstructured"
    
    # --- amenity + type ---
    if isinstance(obj, list) and len(obj) > 0 and all(isinstance(x, dict) for x in obj):
        has_amenity = all('amenity' in x for x in obj)
        has_type = all('type' in x for x in obj)
        if has_amenity and has_type:
            return "structured"
    
    # --- unstructured ---
    return "unstructured"

# classify
df["amenity_group"] = df["room_amenities"].apply(classify)

# 3. display result
counts = df["amenity_group"].value_counts()
total = len(df)

print(counts, "\n")
print("Số dòng có dạng amenity/type rõ ràng (structured):", counts.get("structured", 0))
print("Số dòng có dữ liệu nhưng không đúng dạng (unstructured):", counts.get("unstructured", 0))
print("Số dòng NULL / rỗng (null):", counts.get("null", 0))
print("\nTổng số dòng:", total)

amenity_group
structured      5980
unstructured    4006
null              80
Name: count, dtype: int64 

Số dòng có dạng amenity/type rõ ràng (structured): 5980
Số dòng có dữ liệu nhưng không đúng dạng (unstructured): 4006
Số dòng NULL / rỗng (null): 80

Tổng số dòng: 10066


In [4]:
# Divide into 3 groups

df_structured = df[df["amenity_group"] == "structured"].copy()
df_unstructured = df[df["amenity_group"] == "unstructured"].copy()
df_null        = df[df["amenity_group"] == "null"].copy()

print("Structured:", df_structured.shape)
print("Unstructured:", df_unstructured.shape)
print("Null:", df_null.shape)


Structured: (5980, 3)
Unstructured: (4006, 3)
Null: (80, 3)


In [5]:
df_structured

,room_type_id,room_amenities,amenity_group
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",structured
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",structured
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",structured
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",structured
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",structured
...,...,...,...
10061,10062,"[{'amenity': 'Diện tích phòng: 16 m²', 'type':...",structured
10062,10063,"[{'amenity': 'Diện tích phòng: 16 m²', 'type':...",structured
10063,10064,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",structured
10064,10065,"[{'amenity': 'Diện tích phòng: 15 m²', 'type':...",structured


In [6]:
df_unstructured

,room_type_id,room_amenities,amenity_group
295,296,"['Hướng Ngoài trời', 'phòng tắm riêng', 'Điều ...",unstructured
296,297,"['2 phòng ngủ', '2 phòng tắm', 'Máy sấy tóc', ...",unstructured
297,298,"['Hướng Ngoài trời', 'Ấm nước điện', 'phòng tắ...",unstructured
298,299,"['Hướng Ngoài trời', 'Ấm nước điện', 'phòng tắ...",unstructured
299,300,"['Tối đa 2 người lớn', '1 giường đôi lớn']",unstructured
...,...,...,...
10056,10057,"['Hướng Ngoài trời', 'Ấm nước điện', 'phòng tắ...",unstructured
10057,10058,"['Ấm nước điện', 'phòng tắm riêng', 'Máy sấy t...",unstructured
10058,10059,"['Hướng Ngoài trời', 'Ấm nước điện', 'Bồn tắm'...",unstructured
10059,10060,"['Studio/1 phòng ngủ', '1 phòng tắm', 'Wi-Fi [...",unstructured


In [7]:
df_null

,room_type_id,room_amenities,amenity_group
37,38,[],null
40,41,[],null
587,588,[],null
588,589,[],null
888,889,[],null
...,...,...,...
9366,9367,[],null
9741,9742,[],null
9742,9743,[],null
9748,9749,[],null


In [8]:
import ast
from collections import Counter, defaultdict
import pandas as pd

types = []
# dict: type -> list amenity
type_to_amenities = defaultdict(list)

for s in df_structured["room_amenities"]:
    # parse chuỗi thành list[dict]
    obj = ast.literal_eval(str(s))
    for item in obj:
        t = item.get("type")
        a = item.get("amenity")
        if pd.notna(t) and t != "NULL":
            t = str(t)
            types.append(t)
            if pd.notna(a):
                type_to_amenities[t].append(str(a))

counter = Counter(types)

print("Tổng số loại type khác nhau:", len(counter))
print("\nTổng quan từng type:")
print("(type, số lần xuất hiện, số amenity khác nhau)\n")

for t, c in counter.most_common():
    distinct_amen = len(set(type_to_amenities[t]))
    print(f"{t:25s} {c:5d}  |  {distinct_amen:4d} amenity khác nhau")

# =============================
# List each AMENITY for each TYPE
# =============================

print("\n\nCHI TIẾT TỪNG TYPE:\n")

for t, c in counter.most_common():       # search each type
    amen_counter = Counter(type_to_amenities[t])
    print(f"=== TYPE: {t} ===")
    print(f"Tổng số lần xuất hiện: {c}")
    print(f"Số amenity khác nhau: {len(amen_counter)}")
    print("Danh sách amenity:")

    for amen, cnt in amen_counter.most_common():
        print(f"  - {amen}  --> {cnt}")
    print()   # \n


Tổng số loại type khác nhau: 37

Tổng quan từng type:
(type, số lần xuất hiện, số amenity khác nhau)

confirmation-instant       7442  |    14 amenity khác nhau
bed                        5757  |   578 amenity khác nhau
sqm                        5482  |   166 amenity khác nhau
bathrooms                  4671  |    17 amenity khác nhau
views                      4403  |    25 amenity khác nhau
balcony-terrace            2752  |     1 amenity khác nhau
non-smoking-room           2473  |     1 amenity khác nhau
closet                     2206  |     1 amenity khác nhau
air-conditioning           2093  |     1 amenity khác nhau
mini-bar                   2003  |     1 amenity khác nhau
blackout-curtains          1898  |     1 amenity khác nhau
hair-dryer                  988  |     1 amenity khác nhau
extra-long-beds             946  |     1 amenity khác nhau
complimentary-bottled-water   872  |     1 amenity khác nhau
bathtub                     872  |     1 amenity khác nhau
bedroom    

# Mapping

In [9]:
import ast
import unicodedata
import pandas as pd

# remove sign + tolower -> keyword
def normalize(text: str) -> str:
    s = unicodedata.normalize("NFKD", str(text))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.lower().replace("đ", "d")

# type -> list keyword 
TYPE_KEYWORDS = {
    "bed": [
        "giuong",          
        "nem futon",       
        "bunk bed",
    ],
    "sqm": [
        "dien tich phong", 
        "m2",
        "m²",
    ],
    "bathrooms": [
        "phong tam",       
        "wc",
        "nha ve sinh",
    ],
    "views": [
        "huong thanh pho",
        "huong ngoai troi",
        "huong bien",
        "huong nui",
        "view",
        "huong ho",
        "huong vuon",
        "huong"
    ],
    "balcony-terrace": [
        "ban cong",
        "san hien",
        "ban cong/san hien",
    ],
    "non-smoking-room": [
        "khong hut thuoc",
        "phong khong hut thuoc",
    ],
    "closet": [
        "tu quan ao",
        "tu ao quan",
    ],
    "air-conditioning": [
        "dieu hoa",
        "may lanh",
        "air conditioning",
    ],
    "mini-bar": [
        "tu lanh nho",
        "mini bar",
        "minibar",
    ],
    "blackout-curtains": [
        "rem che anh sang",
        "rem chan sang",
    ],
    "hair-dryer": [
        "may say toc",
        "hair dryer",
    ],
    "extra-long-beds": [
        "giuong cuc dai",
        "giuong sieu dai",
        "extra long bed",
    ],
    "complimentary-bottled-water": [
        "nuoc dong chai mien phi",
        "bottled water",
    ],
    "bathtub": [
        "bon tam ",
        "bon tắm ",   
        "bathtub",
        "bon"
    ],
    "bedroom": [
        "phong ngu",
        "phong ngủ",
        "bedroom",
        "studio/1 phong ngu",
    ],
    "shower": [
        "voi sen",
        "shower",
    ],
    "separate-shower-and-tub": [
        "bon tam/voi sen rieng",
        "bon tam/vòi sen rieng",
        "separate shower and tub",
    ],
    "refrigerator": [
        "tu lanh",
        "refrigerator",
    ],
    "high-floor": [
        "tang cao",
        "high floor",
    ],
    "dressing-room": [
        "phong thay do",
        "dressing room",
    ],
    "ground-floor": [
        "tang tret",
        "tang triệt",
        "ground floor",
    ],
    "private-pool": [
        "be boi rieng",
        "ho boi rieng",
        "private pool",
    ],
    "executive-lounge-access": [
        "phong cho thuong gia",
        "executive lounge",
    ],
    "top-floor": [
        "tang thuong",
        "top floor",
    ],
    "complimentary-instant-coffee": [
        "ca phe hoa tan mien phi",
        "instant coffee mien phi",
    ],
    "jacuzzi-bathtub": [
        "bon tam tao song",
        "jacuzzi",
    ],
    "smoking-allowed": [
        "cho phep hut thuoc",
        "smoking allowed",
    ],
    "electric-blanket": [
        "chan dien",
        "electric blanket",
    ],
    "free-welcome-drink": [
        "do uong moi khach mien phi",
        "welcome drink mien phi",
    ],
    "low-floor": [
        "tang thap",
        "low floor",
    ],
    "wifi": [
        "wi-fi",
        "wifi",
        "truy cap internet - khong day",
    ],
    "complimentary-tea": [
        "tra mien phi",
        "complimentary tea",
    ],
    "coffee-tea-maker": [
        "may pha tra/ca phe",
        "coffee/tea maker",
    ],
    "air-bath-access": [
        "bon tam lo thien",
        "air bath",
    ],
    "hot-spring-access": [
        "suoi nuoc nong",
        "hot spring",
    ],
    "internet": [
        "internet mang lan",
        "mang lan",
        "internet ",
    ],
    "capacity" : [
        "mien phi",
        "toi da",
    ],
    "window": [
        "cua so",
        "window",
    ],
}


In [ ]:
print(normalize("Bồn tắm"))  # điện tich phong

In [10]:
def infer_type_from_amenity(amenity: str):
    """
    Nhận chuỗi amenity (Tiếng Việt) và trả về type (str) hoặc None nếu không đoán được.
    """
    if not isinstance(amenity, str):
        return None

    text = normalize(amenity)

    matched_types = []

    for t, keywords in TYPE_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                matched_types.append(t)
                break  

    if not matched_types:
        return None

    # If match >1 type, use this prio (just in case):
    priority = [
        "sqm", "bed", "bathrooms", "bedroom",
        "views", "balcony-terrace", "non-smoking-room",
    ]
    for t in priority:
        if t in matched_types:
            return t

    # No prio type, use 1st type
    return matched_types[0]


In [11]:
import math

def map_room_amenities_cell(cell):
    """
    cell: chuỗi dạng list (JSON-like) chứa amenity.
    Trả về list các dict: {"amenity": ..., "type": ...}
    """
    if pd.isna(cell) or (isinstance(cell, float) and math.isnan(cell)):
        return cell

    # Parse string -> list Python
    text = str(cell)
    try:
        objs = ast.literal_eval(text)
    except Exception:
        # fail to parse -> cell = 1 amenity string
        objs = [text]

    # if not list -> convert to list
    if not isinstance(objs, (list, tuple)):
        objs = [objs]

    new_objs = []

    for item in objs:
        if isinstance(item, dict):
            amen = item.get("amenity")
            typ = item.get("type")

            # If type = NULL / empty -> guess more
            if (typ is None) or (isinstance(typ, str) and typ.strip().upper() in ("", "NULL")):
                guessed = infer_type_from_amenity(amen)
                if guessed is not None:
                    item["type"] = guessed

            new_objs.append(item)

        else:
            amen = str(item)
            guessed = infer_type_from_amenity(amen)
            new_objs.append({
                "amenity": amen,
                "type": guessed,   # None if cannot guess
            })

    return new_objs

In [12]:
df_unstructured["room_amenities"] = (
    df_unstructured["room_amenities"].apply(map_room_amenities_cell)
)
df_unstructured

,room_type_id,room_amenities,amenity_group
295,296,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
296,297,"[{'amenity': '2 phòng ngủ', 'type': 'bedroom'}...",unstructured
297,298,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
298,299,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
299,300,"[{'amenity': 'Tối đa 2 người lớn', 'type': 'ca...",unstructured
...,...,...,...
10056,10057,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
10057,10058,"[{'amenity': 'Ấm nước điện', 'type': None}, {'...",unstructured
10058,10059,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
10059,10060,"[{'amenity': 'Studio/1 phòng ngủ', 'type': 'be...",unstructured


In [13]:
df_unstructured.to_csv("temp.csv", index=False)

In [14]:
import ast
from collections import Counter, defaultdict
import pandas as pd

types = []
# dict: type -> list amenity
type_to_amenities = defaultdict(list)

for s in df_unstructured["room_amenities"]:
    # parse chuỗi thành list[dict]
    obj = ast.literal_eval(str(s))
    for item in obj:
        t = item.get("type")
        a = item.get("amenity")
        if pd.notna(t) and t != "NULL":
            t = str(t)
            types.append(t)
            if pd.notna(a):
                type_to_amenities[t].append(str(a))

counter = Counter(types)

print("Tổng số loại type khác nhau:", len(counter))
print("\nTổng quan từng type:")
print("(type, số lần xuất hiện, số amenity khác nhau)\n")

for t, c in counter.most_common():
    distinct_amen = len(set(type_to_amenities[t]))
    print(f"{t:25s} {c:5d}  |  {distinct_amen:4d} amenity khác nhau")

# =============================
# List each AMENITY for each TYPE
# =============================

print("\n\nCHI TIẾT TỪNG TYPE:\n")

for t, c in counter.most_common():       # search each type
    amen_counter = Counter(type_to_amenities[t])
    print(f"=== TYPE: {t} ===")
    print(f"Tổng số lần xuất hiện: {c}")
    print(f"Số amenity khác nhau: {len(amen_counter)}")
    print("Danh sách amenity:")

    for amen, cnt in amen_counter.most_common():
        print(f"  - {amen}  --> {cnt}")
    print()   # \n

Tổng số loại type khác nhau: 32

Tổng quan từng type:
(type, số lần xuất hiện, số amenity khác nhau)

bathrooms                  4539  |    18 amenity khác nhau
capacity                   3987  |   150 amenity khác nhau
bed                        3938  |   367 amenity khác nhau
sqm                        3554  |   114 amenity khác nhau
hair-dryer                 2962  |     1 amenity khác nhau
views                      2603  |    25 amenity khác nhau
air-conditioning           2556  |     2 amenity khác nhau
wifi                       1926  |     2 amenity khác nhau
shower                     1868  |     1 amenity khác nhau
refrigerator               1410  |     1 amenity khác nhau
complimentary-bottled-water  1359  |     1 amenity khác nhau
closet                     1126  |     1 amenity khác nhau
mini-bar                   1092  |     1 amenity khác nhau
balcony-terrace             980  |     1 amenity khác nhau
blackout-curtains           961  |     1 amenity khác nhau
bathtub    

In [15]:
df_unstructured.to_csv(
    "room_unstructured_mapped.csv",
    index=False,
    encoding="utf-8-sig"   # để mở bằng Excel không lỗi dấu
)

In [18]:
explode_unstructured = df_unstructured.explode("room_amenities")
print(explode_unstructured)

       room_type_id                                     room_amenities  \
295             296   {'amenity': 'Hướng Ngoài trời', 'type': 'views'}   
295             296  {'amenity': 'phòng tắm riêng', 'type': 'bathro...   
295             296  {'amenity': 'Điều hòa', 'type': 'air-condition...   
295             296                {'amenity': '25 m²', 'type': 'sqm'}   
295             296  {'amenity': 'Tối đa 2 người lớn', 'type': 'cap...   
...             ...                                                ...   
10060         10061    {'amenity': 'Wi-Fi [miễn phí]', 'type': 'wifi'}   
10060         10061  {'amenity': 'Điều hòa', 'type': 'air-condition...   
10060         10061                {'amenity': '30 m²', 'type': 'sqm'}   
10060         10061  {'amenity': 'Tối đa 2 người lớn', 'type': 'cap...   
10060         10061     {'amenity': '1 giường đôi lớn', 'type': 'bed'}   

      amenity_group  
295    unstructured  
295    unstructured  
295    unstructured  
295    unstructured  
2

In [19]:
explode_unstructured["type"] = explode_unstructured["room_amenities"].apply(lambda x: x.get("type") if isinstance(x, dict) else None)
explode_unstructured["amenity"] = explode_unstructured["room_amenities"].apply(lambda x: x.get("amenity") if isinstance(x, dict) else None)

In [20]:
print(explode_unstructured["type"])

295                 views
295             bathrooms
295      air-conditioning
295                   sqm
295              capacity
               ...       
10060                wifi
10060    air-conditioning
10060                 sqm
10060            capacity
10060                 bed
Name: type, Length: 42467, dtype: object


In [21]:
none_type_df = explode_unstructured[explode_unstructured["type"].isna()]

In [22]:
print(none_type_df)

       room_type_id                                     room_amenities  \
297             298          {'amenity': 'Ấm nước điện', 'type': None}   
297             298  {'amenity': 'Đồ dùng cho giấc ngủ thoải mái', ...   
297             298    {'amenity': 'Trái cây/đồ ăn vặt', 'type': None}   
298             299          {'amenity': 'Ấm nước điện', 'type': None}   
298             299  {'amenity': 'Đồ dùng cho giấc ngủ thoải mái', ...   
...             ...                                                ...   
10045         10046               {'amenity': 'Ấm nước', 'type': None}   
10051         10052               {'amenity': 'Ấm nước', 'type': None}   
10056         10057          {'amenity': 'Ấm nước điện', 'type': None}   
10057         10058          {'amenity': 'Ấm nước điện', 'type': None}   
10058         10059          {'amenity': 'Ấm nước điện', 'type': None}   

      amenity_group  type                         amenity  
297    unstructured  None                    Ấm nướ

In [23]:
none_type_df.to_csv(
    "room_unstructured_none_type.csv",
    index=False,
    encoding="utf-8-sig"   # để mở bằng Excel không lỗi dấu
)

In [24]:
explode_unstructured.loc[explode_unstructured["type"].isna(), ["amenity"]].drop_duplicates().sort_values("amenity").to_csv(
    "room_unstructured_none_type_amenity.csv",index=False,)

In [ ]:
none_type = pd.read_csv("room_unstructured_none_type_amenity.csv")
none_type["normalized_amenity"] = none_type["amenity"].apply(normalize)
print(none_type)

                                     amenity  \
0                            Hành lang ngoài   
1                     Lối vào bãi biển riêng   
2   Máy cà phê espresso có viên nén (có phí)   
3                             Phòng xông khô   
4                                       Quạt   
5                                       Rượu   
6                                       Sưởi   
7                         Trái cây/đồ ăn vặt   
8             Đồ dùng cho giấc ngủ thoải mái   
9                           Đồ gỗ ngoài trời   
10                                   Ấm nước   
11                              Ấm nước điện   

                          normalized_amenity  
0                            hanh lang ngoai  
1                     loi vao bai bien rieng  
2   may ca phe espresso co vien nen (co phi)  
3                             phong xong kho  
4                                       quat  
5                                       ruou  
6                                       suoi  

In [27]:
df_unstructured

,room_type_id,room_amenities,amenity_group
295,296,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
296,297,"[{'amenity': '2 phòng ngủ', 'type': 'bedroom'}...",unstructured
297,298,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
298,299,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
299,300,"[{'amenity': 'Tối đa 2 người lớn', 'type': 'ca...",unstructured
...,...,...,...
10056,10057,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
10057,10058,"[{'amenity': 'Ấm nước điện', 'type': None}, {'...",unstructured
10058,10059,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
10059,10060,"[{'amenity': 'Studio/1 phòng ngủ', 'type': 'be...",unstructured


In [28]:
# mapping bổ sung cho TYPE_KEYWORDS
#  --- IGNORE ---
TYPE_KEYWORDS.update(
    {
        "confirmation-instant" : [
        "hanh lang ngoai" ,
        "loi vao bai bien rieng"  ,
        "may ca phe espresso co vien nen (co phi)"  ,
        "phong xong kho"  ,
        "quat"  ,
        "ruou"  ,
        "suoi"  ,
        "trai cay/do an vat"  ,
        "do dung cho giac ngu thoai mai"  ,
        "do go ngoai troi"  ,
        "am nuoc"  ,
        "am nuoc dien" ,
        ]
    }
)

In [32]:
def infer_type_from_amenity(amenity: str):
    """
    Nhận chuỗi amenity (Tiếng Việt) và trả về type (str) hoặc None nếu không đoán được.
    """
    if not isinstance(amenity, str):
        return None

    text = normalize(amenity)

    matched_types = []

    for t, keywords in TYPE_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                matched_types.append(t)
                break  

    if not matched_types:
        return None

    # If match >1 type, use this prio (just in case):
    priority = [
        "sqm", "bed", "bathrooms", "bedroom",
        "views", "balcony-terrace", "non-smoking-room",
    ]
    for t in priority:
        if t in matched_types:
            return t

    # No prio type, use 1st type
    return matched_types[0]


In [33]:
import math

def map_room_amenities_cell(cell):
    """
    cell: chuỗi dạng list (JSON-like) chứa amenity.
    Trả về list các dict: {"amenity": ..., "type": ...}
    """
    if pd.isna(cell) or (isinstance(cell, float) and math.isnan(cell)):
        return cell

    # Parse string -> list Python
    text = str(cell)
    try:
        objs = ast.literal_eval(text)
    except Exception:
        # fail to parse -> cell = 1 amenity string
        objs = [text]

    # if not list -> convert to list
    if not isinstance(objs, (list, tuple)):
        objs = [objs]

    new_objs = []

    for item in objs:
        if isinstance(item, dict):
            amen = item.get("amenity")
            typ = item.get("type")

            # If type = NULL / empty -> guess more
            if (typ is None) or (isinstance(typ, str) and typ.strip().upper() in ("", "NULL")):
                guessed = infer_type_from_amenity(amen)
                if guessed is not None:
                    item["type"] = guessed

            new_objs.append(item)

        else:
            amen = str(item)
            guessed = infer_type_from_amenity(amen)
            new_objs.append({
                "amenity": amen,
                "type": guessed,   # None if cannot guess
            })

    return new_objs

In [44]:
def update_room_amenities(amenities):
    for amenity in amenities:
        if amenity.get("type") is None:
            guessed = infer_type_from_amenity(amenity.get("amenity"))
            if guessed is not None:
                amenity["type"] = guessed
    return amenities

In [45]:
df_unstructured_v2 = df_unstructured.copy()
df_unstructured_v2["room_amenities"] = df_unstructured_v2["room_amenities"].apply(update_room_amenities)

In [46]:
df_unstructured_v2.to_csv(
    "room_unstructured_mapped_v2.csv",
    index=False,
    encoding="utf-8-sig"   # để mở bằng Excel không lỗi dấu
)

In [47]:
df_unstructured_v2.drop(columns=["amenity_group"], inplace=True)

In [48]:
df_unstructured_v2

,room_type_id,room_amenities
295,296,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."
296,297,"[{'amenity': '2 phòng ngủ', 'type': 'bedroom'}..."
297,298,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."
298,299,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."
299,300,"[{'amenity': 'Tối đa 2 người lớn', 'type': 'ca..."
...,...,...
10056,10057,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."
10057,10058,"[{'amenity': 'Ấm nước điện', 'type': 'confirma..."
10058,10059,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."
10059,10060,"[{'amenity': 'Studio/1 phòng ngủ', 'type': 'be..."


In [50]:
df_structured.drop(columns=["amenity_group"], inplace=True)

In [51]:
df_structured

,room_type_id,room_amenities
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':..."
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':..."
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':..."
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':..."
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':..."
...,...,...
10061,10062,"[{'amenity': 'Diện tích phòng: 16 m²', 'type':..."
10062,10063,"[{'amenity': 'Diện tích phòng: 16 m²', 'type':..."
10063,10064,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':..."
10064,10065,"[{'amenity': 'Diện tích phòng: 15 m²', 'type':..."


In [52]:
merge_df = pd.concat([df_structured, df_unstructured_v2, df_null], ignore_index=True)

In [53]:
merge_df.to_csv(
    "room_amenities_cleaned.csv",
    index=False,
    encoding="utf-8-sig"   # để mở bằng Excel không lỗi dấu
)